# Sentiment Analysis of Amazon Reviews

This notebook demonstrates how to fine-tune a BERT model for sentiment classification on the Amazon customer reviews dataset.

## 1. Setup

First, let's install the necessary libraries.

In [1]:
!pip install transformers datasets scikit-learn torch

## 2. Load and Preprocess the Dataset

In [2]:
from datasets import load_dataset

# 1. Load
dataset = load_dataset("SetFit/amazon_reviews_multi_en", split="train")

# 2. Split
dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset["train"]
test_dataset = dataset["test"]

# 3. Remap labels → (important !)
def clean_labels(example):
    if example["label"] in [0, 1]:       # negative
        return {"label": 0}
    elif example["label"] == 2:          # neutral
        return {"label": 1}
    else:                                # 3,4 → positive
        return {"label": 2}

train_dataset = train_dataset.map(clean_labels)
test_dataset = test_dataset.map(clean_labels)

print("Labels uniques train:", set(train_dataset["label"]))
print("Labels uniques test:", set(test_dataset["label"]))


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Labels uniques train: {0, 1, 2}
Labels uniques test: {0, 1, 2}


## 3. Tokenization

In [3]:
from transformers import AutoTokenizer

model_name = 'google-bert/bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True)

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])
test_dataset.set_format('torch', columns=['input_ids', 'attention_mask', 'label'])


## 4. Model Fine-Tuning

In [4]:
from transformers import AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

# Définition des labels
label2id = {'negative': 0, 'neutral': 1, 'positive': 2}
id2label = {0: 'negative', 1: 'neutral', 2: 'positive'}

# Chargement du modèle
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

# Fonction pour calculer métriques
def compute_metrics(pred):
    labels = pred.label_ids
    preds = np.argmax(pred.predictions, axis=1)
    f1 = f1_score(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'f1': f1}

# Arguments d'entraînement optimisés pour Colab GPU
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=1,                   # 1 epoch pour test rapide
    per_device_train_batch_size=32,       # batch plus grand grâce au GPU
    per_device_eval_batch_size=32,
    warmup_steps=100,                     # échauffement du scheduler
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    eval_strategy='epoch',                # évaluation à chaque epoch
    fp16=True                             # accélération GPU (half-precision)
)

# Création du Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

# Lancement de l'entraînement
trainer.train()


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
wandb: Currently logged in as: aristondecor003 (aristondecor003-faculty-of-science-and-technics-of-fes) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.552900,0.504570,0.785650,0.723567


TrainOutput(global_step=5000, training_loss=0.5513618492126465, metrics={'train_runtime': 4094.0083, 'train_samples_per_second': 39.082, 'train_steps_per_second': 1.221, 'total_flos': 4.209814683648e+16, 'train_loss': 0.5513618492126465, 'epoch': 1.0})

## 5. Evaluation

In [5]:
eval_results = trainer.evaluate()
print(f"Evaluation results: {eval_results}")

Evaluation results: {'eval_loss': 0.5045698881149292, 'eval_accuracy': 0.78565, 'eval_f1': 0.7235669069578314, 'eval_runtime': 303.2589, 'eval_samples_per_second': 131.901, 'eval_steps_per_second': 4.122, 'epoch': 1.0}


## 6. Prediction on New Reviews

In [6]:
from transformers import pipeline

# Create a prediction pipeline
sentiment_pipeline = pipeline('sentiment-analysis', model=model, tokenizer=tokenizer)

# Sample reviews
new_reviews = [
    "This product is amazing! I love it.",
    "The product is okay, not great but not terrible.",
    "I'm very disappointed with this purchase."
]

# Get predictions
predictions = sentiment_pipeline(new_reviews)

# The pipeline now directly outputs the sentiment label
for review, pred in zip(new_reviews, predictions):
    print(f'Review: "{review}" -> Sentiment: {pred["label"]} (Score: {pred["score"]:.4f})')


Device set to use cuda:0


Review: "This product is amazing! I love it." -> Sentiment: positive (Score: 0.9917)
Review: "The product is okay, not great but not terrible." -> Sentiment: neutral (Score: 0.7549)
Review: "I'm very disappointed with this purchase." -> Sentiment: negative (Score: 0.9577)
